# STC Jawwy

In [ ]:
"""
Here we install libraries that are not installed by default
Example:  pyslsb
Feel free to add any library you are planning to use.
"""
!pip install pyxlsb

In [ ]:
# Import the required libraries
"""
Please feel free to import any required libraries as per your needs
"""
import pandas as pd     # provides high-performance, easy to use structures and data analysis tools
import pyxlsb           # Excel extention to read xlsb files (the input file)
import numpy as np      # provides fast mathematical computation on arrays and matrices


# Jawwy dataset
The dataset consists of details about each customer and the movies and/or tv shows watched in addition to the genre.

You are required to work on task three to build a recommendation engine for our platform to Recommend movies to usesrs that they might be interested in¶


In [ ]:
dataframe = pd.read_excel("/content/sample_data/stc TV Data Set_T3.xlsx",index_col=0)
# Please make a copy of dataset if you are going to work directly and make changes on the dataset
# you can use   df=dataframe.copy()

In [ ]:
# check the data shape
dataframe.shape

(1048575, 5)

In [ ]:
# display the first 5 rows
dataframe.head()

,user_id_maped,program_name,rating,date_,program_genre
0,26138,100 treets,1,2017-05-27,Drama
1,7946,Moana,1,2017-05-21,Animation
2,7418,The Mermaid Princess,1,2017-08-10,Animation
3,19307,The Mermaid Princess,2,2017-07-26,Animation
4,15860,Churchill,2,2017-07-07,Biography


In [ ]:
# describe the numeric values in the dataset
dataframe.describe()

,user_id_maped,rating,date_
count,1.048575e+06,1.048575e+06,1048575
mean,1.709266e+04,2.497283e+00,2017-10-04 00:23:20.346183936
min,1.000000e+00,1.000000e+00,2017-03-14 00:00:00
25%,8.253000e+03,1.000000e+00,2017-06-10 00:00:00
50%,1.714900e+04,2.000000e+00,2017-10-14 00:00:00
75%,2.566500e+04,3.000000e+00,2018-01-21 00:00:00
max,3.428000e+04,4.000000e+00,2018-04-30 00:00:00
std,1.003513e+04,1.119837e+00,NaN


In [ ]:
# check if any column has null value in the dataset
dataframe.isnull().any()

,0
user_id_maped,False
program_name,False
rating,False
date_,False
program_genre,False


In [ ]:
# we import Visualization libraries
# you can ignore and use any other graphing libraries
import matplotlib.pyplot as plt # a comprehensive library for creating static, animated, and interactive visualizations
import plotly #a graphing library makes interactive, publication-quality graphs. Examples of how to make line plots, scatter plots, area charts, bar charts, error bars, box plots, histograms, heatmaps, subplots, multiple-axes, polar charts, and bubble charts.
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
"""
TODO build your Recommender system to Highlight Programs that usesrs might be interested in
"""

'\nTODO build your Recommender system to Highlight Programs that usesrs might be interested in\n'

In [ ]:
"""
TODO show the recommendations (top 5) for the people who watched "Moana" movie
"""

'\nTODO show the recommendations (top 5) for the people who watched "Moana" movie\n'

In [ ]:
# Prepare data for recommendation system using original ratings
df_rec = dataframe.copy()
df_rec = df_rec.dropna()
print(f"Data shape after cleaning: {df_rec.shape}")
df_rec.head()

Data shape after cleaning: (1048575, 5)


,user_id_maped,program_name,rating,date_,program_genre
0,26138,100 treets,1,2017-05-27,Drama
1,7946,Moana,1,2017-05-21,Animation
2,7418,The Mermaid Princess,1,2017-08-10,Animation
3,19307,The Mermaid Princess,2,2017-07-26,Animation
4,15860,Churchill,2,2017-07-07,Biography


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Create user-item matrix using original ratings (1-4)
user_item_matrix = df_rec.pivot_table(index='user_id_maped',
                                       columns='program_name',
                                       values='rating',
                                       fill_value=0)

print(f"User-Item Matrix shape: {user_item_matrix.shape}")
print(f"Number of users: {user_item_matrix.shape[0]}")
print(f"Number of programs: {user_item_matrix.shape[1]}")

User-Item Matrix shape: (11578, 8013)
Number of users: 11578
Number of programs: 8013


In [ ]:
# Calculate cosine similarity between programs
item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity,
                                   index=user_item_matrix.columns,
                                   columns=user_item_matrix.columns)

print("Item similarity matrix shape:", item_similarity_df.shape)
print("\nTop 5 programs most similar to 'Moana':")
if 'Moana' in item_similarity_df.columns:
    print(item_similarity_df['Moana'].sort_values(ascending=False).head(6))

Item similarity matrix shape: (8013, 8013)

Top 5 programs most similar to 'Moana':
program_name
Moana                                    1.000000
Trolls                                   0.572358
Surf's Up : WaveMania                    0.529424
The Mermaid Princess                     0.493362
The Boss Baby                            0.448557
The Jetsons & WWE: Robo-WrestleMania!    0.438942
Name: Moana, dtype: float64


In [ ]:
def recommend_similar_programs(program_name, similarity_df, n_recommendations=5):
    """Recommend programs similar to a given program"""
    if program_name not in similarity_df.columns:
        return f"Program '{program_name}' not found"

    similar = similarity_df[program_name].sort_values(ascending=False)
    similar = similar.drop(program_name)
    return similar.head(n_recommendations)

def recommend_for_user(user_id, user_item_matrix, item_similarity_df, n_recommendations=5):
    """Recommend programs for a user based on their rated programs"""
    if user_id not in user_item_matrix.index:
        return f"User {user_id} not found"

    user_ratings = user_item_matrix.loc[user_id]
    watched = user_ratings[user_ratings > 0].index.tolist()

    if len(watched) == 0:
        return f"User {user_id} has no ratings"

    scores = {}
    for program in watched:
        rating = user_ratings[program]
        similar = item_similarity_df[program]

        for sim_prog, sim_score in similar.items():
            if sim_prog not in watched:
                scores[sim_prog] = scores.get(sim_prog, 0) + (rating * sim_score)

    recommendations = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return recommendations[:n_recommendations]

print("Recommendation functions loaded successfully!")

Recommendation functions loaded successfully!


In [ ]:
# Find users who watched Moana
moana_data = df_rec[df_rec['program_name'] == 'Moana']
moana_users = moana_data['user_id_maped'].unique()
print(f"Found {len(moana_users)} users who watched Moana")
print(f"Moana ratings distribution: {moana_data['rating'].value_counts().sort_index()}\n")

# Show recommendations for first 5 users who have other ratings
count = 0
for user in moana_users:
    if count >= 5:
        break

    user_other = df_rec[(df_rec['user_id_maped'] == user) & (df_rec['program_name'] != 'Moana')]
    if len(user_other) >= 2:  # User has at least 2 other ratings
        moana_rating = moana_data[moana_data['user_id_maped'] == user]['rating'].values[0]
        recs = recommend_for_user(user, user_item_matrix, item_similarity_df, 5)

        print(f"User {user} (rated Moana: {moana_rating}/4):")
        for prog, score in recs:
            print(f"   → {prog} (score: {score:.3f})")
        print()
        count += 1

if count == 0:
    print("No Moana viewers have enough ratings for personalized recommendations.")
    print("Using program similarity instead (see Cell #16).")

Found 2173 users who watched Moana
Moana ratings distribution: rating
1    1426
2    5377
3     460
4     818
Name: count, dtype: int64

User 7946 (rated Moana: 1/4):
   → Trolls (score: 0.943)
   → Surf's Up : WaveMania (score: 0.905)
   → The Mermaid Princess (score: 0.849)
   → The Boss Baby (score: 0.832)
   → Assassin's Creed (score: 0.803)

User 33114 (rated Moana: 2/4):
   → The Mermaid Princess (score: 3.245)
   → The Boss Baby (score: 3.035)
   → The Jetsons & WWE: Robo-WrestleMania! (score: 3.033)
   → The Birth of a Nation (score: 2.741)
   → Storks (score: 2.714)

User 27997 (rated Moana: 2/4):
   → The Mermaid Princess (score: 2.632)
   → The Jetsons & WWE: Robo-WrestleMania! (score: 2.391)
   → The Boss Baby (score: 2.342)
   → Storks (score: 2.065)
   → Rings (score: 1.890)

User 30363 (rated Moana: 2/4):
   → The Mermaid Princess (score: 18.467)
   → The Little Vampire (score: 16.494)
   → The Jetsons & WWE: Robo-WrestleMania! (score: 16.365)
   → War for the Planet of 

In [ ]:
from collections import Counter

# Limit to first 100 users for faster execution
print(f"Total Moana viewers: {len(moana_users)}")
print("Limiting to first 100 users for faster recommendations...")

all_recs = []
users_processed = 0

for user in moana_users[:100]:  # ← Only process 100 users instead of all 2173
    user_other = df_rec[(df_rec['user_id_maped'] == user) & (df_rec['program_name'] != 'Moana')]
    if len(user_other) >= 2:
        recs = recommend_for_user(user, user_item_matrix, item_similarity_df, 10)
        if not isinstance(recs, str):
            for prog, score in recs:
                all_recs.append(prog)
        users_processed += 1

print(f"Processed {users_processed} users with multiple ratings")

if len(all_recs) > 0:
    counter = Counter(all_recs)
    top_5 = counter.most_common(5)

    print("\n" + "="*60)
    print(" TOP 5 RECOMMENDED PROGRAMS FOR PEOPLE WHO WATCHED 'MOANA'")
    print("="*60)
    for prog, count in top_5:
        print(f"   {count} recommendations → {prog}")
else:
    print("\nUsing program similarity approach instead:")
    if 'Moana' in item_similarity_df.columns:
        print("\n TOP 5 PROGRAMS MOST SIMILAR TO 'MOANA':")
        similar = item_similarity_df['Moana'].sort_values(ascending=False).head(6)
        for prog, score in similar.items():
            if prog != 'Moana':
                print(f"   → {prog} (similarity: {score:.4f})")

Total Moana viewers: 2173
Limiting to first 100 users for faster recommendations...
Processed 83 users with multiple ratings

 TOP 5 RECOMMENDED PROGRAMS FOR PEOPLE WHO WATCHED 'MOANA'
   45 recommendations → Rings
   43 recommendations → Alien: Covenant
   39 recommendations → Collateral Beauty
   38 recommendations → The BFG
   38 recommendations → The Jetsons & WWE: Robo-WrestleMania!


In [ ]:
# Final output for sharing - Top 5 recommendations
print("\n" + "="*60)
print(" STC JAWWY TV - RECOMMENDATION ENGINE RESULTS")
print("="*60)

if 'Moana' in item_similarity_df.columns:
    print("\nBased on collaborative filtering (users who watched Moana also liked):")
    similar = item_similarity_df['Moana'].sort_values(ascending=False).head(6)
    rank = 1
    for prog, score in similar.items():
        if prog != 'Moana':
            print(f"   {rank}. {prog} (similarity score: {score:.4f})")
            rank += 1
else:
    print("Moana not found in dataset")

print("\n" + "="*60)
print(" Recommendation system built successfully!")
print("="*60)


 STC JAWWY TV - RECOMMENDATION ENGINE RESULTS

Based on collaborative filtering (users who watched Moana also liked):
   1. Trolls (similarity score: 0.5724)
   2. Surf's Up : WaveMania (similarity score: 0.5294)
   3. The Mermaid Princess (similarity score: 0.4934)
   4. The Boss Baby (similarity score: 0.4486)
   5. The Jetsons & WWE: Robo-WrestleMania! (similarity score: 0.4389)

 Recommendation system built successfully!
